## Imports

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.request
import json


%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")

## Load Data

In [16]:
import urllib.request
import json

def fetch_nvd_cves(start_date, end_date):
    url = (
        f"https://services.nvd.nist.gov/rest/json/cves/2.0"
        f"?resultsPerPage=2000"
        f"&pubStartDate={start_date}T00:00:00.000"
        f"&pubEndDate={end_date}T00:00:00.000"
    )
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as response:
        return json.loads(response.read().decode())

def parse_cves(raw):
    records = []
    for item in raw["vulnerabilities"]:
        cve = item["cve"]
        metrics = cve.get("metrics", {})
        cvss3 = metrics.get("cvssMetricV31", metrics.get("cvssMetricV30", []))
        cvss_data = cvss3[0]["cvssData"] if cvss3 else {}
        records.append({
            "cve_id":                  cve["id"],
            "published":               cve["published"],
            "last_modified":           cve["lastModified"],
            "vuln_status":             cve.get("vulnStatus"),
            "description":             cve["descriptions"][0]["value"] if cve["descriptions"] else None,
            "base_score":              cvss_data.get("baseScore"),
            "base_severity":           cvss_data.get("baseSeverity"),
            "attack_vector":           cvss_data.get("attackVector"),
            "attack_complexity":       cvss_data.get("attackComplexity"),
            "privileges_required":     cvss_data.get("privilegesRequired"),
            "user_interaction":        cvss_data.get("userInteraction"),
            "confidentiality_impact":  cvss_data.get("confidentialityImpact"),
            "integrity_impact":        cvss_data.get("integrityImpact"),
            "availability_impact":     cvss_data.get("availabilityImpact"),
            "exploitability_score":    cvss3[0].get("exploitabilityScore") if cvss3 else None,
        })
    return records

# Two 120-day windows covering most of 2024
print("Fetching window 1...")
raw1 = fetch_nvd_cves("2024-01-01", "2024-04-30")
print("Fetching window 2...")
raw2 = fetch_nvd_cves("2024-05-01", "2024-08-28")

records = parse_cves(raw1) + parse_cves(raw2)
df = pd.DataFrame(records)

print(f"Shape: {df.shape}")
print(f"Rows with CVSS scores: {df['base_score'].notna().sum()}")

Fetching window 1...
Fetching window 2...
Shape: (4000, 15)
Rows with CVSS scores: 3946


In [17]:
df.head()

,cve_id,published,last_modified,vuln_status,description,base_score,base_severity,attack_vector,attack_complexity,privileges_required,user_interaction,confidentiality_impact,integrity_impact,availability_impact,exploitability_score
0,CVE-2024-21732,2024-01-01T08:15:36.087,2025-06-03T15:15:56.083,Modified,FlyCms through abbaa5a allows XSS via the perm...,6.1,MEDIUM,NETWORK,LOW,NONE,REQUIRED,LOW,LOW,NONE,2.8
1,CVE-2023-5877,2024-01-01T15:15:42.727,2025-06-03T15:15:50.457,Modified,The affiliate-toolkit WordPress plugin before ...,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9
2,CVE-2023-6000,2024-01-01T15:15:43.100,2025-06-18T15:15:24.327,Modified,The Popup Builder WordPress plugin before 4.2....,6.1,MEDIUM,NETWORK,LOW,NONE,REQUIRED,LOW,LOW,NONE,2.8
3,CVE-2023-6037,2024-01-01T15:15:43.147,2025-06-18T15:15:24.890,Modified,The WP TripAdvisor Review Slider WordPress plu...,4.8,MEDIUM,NETWORK,LOW,HIGH,REQUIRED,LOW,LOW,NONE,1.7
4,CVE-2023-6064,2024-01-01T15:15:43.197,2025-05-13T15:15:51.197,Modified,The PayHere Payment Gateway WordPress plugin b...,7.5,HIGH,NETWORK,LOW,NONE,NONE,HIGH,NONE,NONE,3.9


In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   cve_id                  4000 non-null   str    
 1   published               4000 non-null   str    
 2   last_modified           4000 non-null   str    
 3   vuln_status             4000 non-null   str    
 4   description             4000 non-null   str    
 5   base_score              3946 non-null   float64
 6   base_severity           3946 non-null   str    
 7   attack_vector           3946 non-null   str    
 8   attack_complexity       3946 non-null   str    
 9   privileges_required     3946 non-null   str    
 10  user_interaction        3946 non-null   str    
 11  confidentiality_impact  3946 non-null   str    
 12  integrity_impact        3946 non-null   str    
 13  availability_impact     3946 non-null   str    
 14  exploitability_score    3946 non-null   float64
dty

In [20]:
df.describe()

,base_score,exploitability_score
count,3946.000000,3946.000000
mean,6.828713,2.484389
std,1.706201,0.920412
min,0.000000,0.200000
25%,5.500000,1.800000
50%,6.900000,2.500000
75%,7.800000,2.800000
max,10.000000,3.900000


## Data Wrangling

In [21]:
# Why: 54 rows have no scores at all — unanalyzable for our purposes
# These are CVEs still awaiting NVD analysis (vuln_status = 'Awaiting Analysis')
print(f"Rows before: {len(df)}")
df = df.dropna(subset=["base_score"]).reset_index(drop=True)
print(f"Rows after: {len(df)}")
print(f"Dropped: {4000 - len(df)} rows")

Rows before: 4000
Rows after: 3946
Dropped: 54 rows


In [22]:
# Why: string dates can't be sorted, filtered by month, or used in time-series plots
df["published"] = pd.to_datetime(df["published"])
df["last_modified"] = pd.to_datetime(df["last_modified"])

# Extract useful time features while we're here
df["published_month"] = df["published"].dt.to_period("M")
df["days_to_modify"] = (df["last_modified"] - df["published"]).dt.days

print(df[["published", "last_modified", "published_month", "days_to_modify"]].head())

                published           last_modified published_month  \
0 2024-01-01 08:15:36.087 2025-06-03 15:15:56.083         2024-01   
1 2024-01-01 15:15:42.727 2025-06-03 15:15:50.457         2024-01   
2 2024-01-01 15:15:43.100 2025-06-18 15:15:24.327         2024-01   
3 2024-01-01 15:15:43.147 2025-06-18 15:15:24.890         2024-01   
4 2024-01-01 15:15:43.197 2025-05-13 15:15:51.197         2024-01   

   days_to_modify  
0             519  
1             519  
2             533  
3             533  
4             498  


In [23]:
# Why: string columns waste memory and don't sort meaningfully
# Ordered categories let us sort severity as LOW < MEDIUM < HIGH < CRITICAL
severity_order = ["LOW", "MEDIUM", "HIGH", "CRITICAL"]
df["base_severity"] = pd.Categorical(df["base_severity"], 
                                      categories=severity_order, 
                                      ordered=True)

# These don't need ordering — just nominal categories
for col in ["attack_vector", "attack_complexity", "privileges_required",
            "user_interaction", "confidentiality_impact", 
            "integrity_impact", "availability_impact", "vuln_status"]:
    df[col] = df[col].astype("category")

print(df.dtypes)

cve_id                               str
published                 datetime64[us]
last_modified             datetime64[us]
vuln_status                     category
description                          str
base_score                       float64
base_severity                   category
attack_vector                   category
attack_complexity               category
privileges_required             category
user_interaction                category
confidentiality_impact          category
integrity_impact                category
availability_impact             category
exploitability_score             float64
published_month                period[M]
days_to_modify                     int64
dtype: object


/var/folders/s1/84dvbf5j7tsdt1bqd52f9d2m0000gn/T/ipykernel_97704/3318961975.py:4: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  df["base_severity"] = pd.Categorical(df["base_severity"],


In [24]:
# Why: we need a binary label to train a classifier
# "High exploitability" = exploitability_score above the median
# This is our way of asking: which CVEs are easiest for an attacker to use?
median_exploit = df["exploitability_score"].median()
df["high_exploitability"] = (df["exploitability_score"] >= median_exploit).astype(int)

print(f"Median exploitability score: {median_exploit}")
print(f"High exploitability (1): {df['high_exploitability'].sum()}")
print(f"Low exploitability  (0): {(df['high_exploitability'] == 0).sum()}")

Median exploitability score: 2.5
High exploitability (1): 2031
Low exploitability  (0): 1915


In [25]:
print(df.shape)
df.info()

(3946, 18)
<class 'pandas.DataFrame'>
RangeIndex: 3946 entries, 0 to 3945
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   cve_id                  3946 non-null   str           
 1   published               3946 non-null   datetime64[us]
 2   last_modified           3946 non-null   datetime64[us]
 3   vuln_status             3946 non-null   category      
 4   description             3946 non-null   str           
 5   base_score              3946 non-null   float64       
 6   base_severity           3945 non-null   category      
 7   attack_vector           3946 non-null   category      
 8   attack_complexity       3946 non-null   category      
 9   privileges_required     3946 non-null   category      
 10  user_interaction        3946 non-null   category      
 11  confidentiality_impact  3946 non-null   category      
 12  integrity_impact        3946 non-null   category

In [26]:
# One row has a base_score but no severity label
# CVSS v3 severity bands: 0.0 LOW, 4.0 MEDIUM, 7.0 HIGH, 9.0 CRITICAL
def score_to_severity(score):
    if score >= 9.0: return "CRITICAL"
    elif score >= 7.0: return "HIGH"
    elif score >= 4.0: return "MEDIUM"
    else: return "LOW"

mask = df["base_severity"].isna()
df.loc[mask, "base_severity"] = df.loc[mask, "base_score"].apply(score_to_severity)

print(f"Missing severities remaining: {df['base_severity'].isna().sum()}")

Missing severities remaining: 0


In [27]:
df.head()

,cve_id,published,last_modified,vuln_status,description,base_score,base_severity,attack_vector,attack_complexity,privileges_required,user_interaction,confidentiality_impact,integrity_impact,availability_impact,exploitability_score,published_month,days_to_modify,high_exploitability
0,CVE-2024-21732,2024-01-01 08:15:36.087,2025-06-03 15:15:56.083,Modified,FlyCms through abbaa5a allows XSS via the perm...,6.1,MEDIUM,NETWORK,LOW,NONE,REQUIRED,LOW,LOW,NONE,2.8,2024-01,519,1
1,CVE-2023-5877,2024-01-01 15:15:42.727,2025-06-03 15:15:50.457,Modified,The affiliate-toolkit WordPress plugin before ...,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,HIGH,HIGH,HIGH,3.9,2024-01,519,1
2,CVE-2023-6000,2024-01-01 15:15:43.100,2025-06-18 15:15:24.327,Modified,The Popup Builder WordPress plugin before 4.2....,6.1,MEDIUM,NETWORK,LOW,NONE,REQUIRED,LOW,LOW,NONE,2.8,2024-01,533,1
3,CVE-2023-6037,2024-01-01 15:15:43.147,2025-06-18 15:15:24.890,Modified,The WP TripAdvisor Review Slider WordPress plu...,4.8,MEDIUM,NETWORK,LOW,HIGH,REQUIRED,LOW,LOW,NONE,1.7,2024-01,533,0
4,CVE-2023-6064,2024-01-01 15:15:43.197,2025-05-13 15:15:51.197,Modified,The PayHere Payment Gateway WordPress plugin b...,7.5,HIGH,NETWORK,LOW,NONE,NONE,HIGH,NONE,NONE,3.9,2024-01,498,1
